# Insulyn AI - Type 2 Diabetes Early Detection

**Authors:** Muhammed Jalahej, Yazen Emino

## 1. Introduction

This notebook documents the machine learning pipeline for **Insulyn AI**, a platform for early detection of Type 2 Diabetes. The goal is to train, evaluate, and explain a binary classifier that predicts diabetes risk from clinical measurements.

**Pipeline overview:**
1. Data loading and exploration (EDA)
2. Data preprocessing (handling missing values)
3. Feature scaling (StandardScaler)
4. Model training and comparison (Random Forest, XGBoost, XGBoost Tuned)
5. Evaluation (Accuracy, Precision, Recall, F1, AUC-ROC, Confusion Matrix, PR Curve)
6. Explainability (SHAP - SHapley Additive exPlanations)
7. Model saving as a deployable bundle

## 2. Problem Statement

Type 2 Diabetes is one of the most prevalent chronic diseases worldwide. According to the **International Diabetes Federation (IDF)**, approximately **537 million adults** (20-79 years) were living with diabetes in 2021, and this number is projected to reach **783 million by 2045**.

### Key challenges:
- A significant portion of people with diabetes remain **undiagnosed**
- Late diagnosis leads to severe complications (cardiovascular disease, kidney failure, blindness)
- Early detection can enable **timely intervention** and better health outcomes

### Objective
Build a machine learning model that can **predict diabetes risk** from readily available clinical measurements such as glucose level, BMI, blood pressure, and age, enabling early screening and intervention.

### Success Metrics
- **High Recall**: Minimize false negatives (missing diabetic patients is dangerous)
- **Balanced F1**: Good trade-off between precision and recall
- **Strong AUC-ROC**: Good discrimination between classes
- **Explainability**: Understand which features drive the prediction

## 3. Setup & Imports

In [ ]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    precision_recall_curve, average_precision_score
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import shap

warnings.filterwarnings('ignore')
sns.set(style='whitegrid', palette='muted', font_scale=1.1)
pd.set_option('display.max_columns', None)

# Configuration
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_SPLITS_CV = 5

FEATURE_NAMES = [
    'pregnancies', 'glucose', 'blood_pressure', 'skin_thickness',
    'insulin', 'bmi', 'diabetes_pedigree_function', 'age'
]

print('All libraries imported successfully.')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'Scikit-learn: {__import__("sklearn").__version__}')
print(f'XGBoost: {__import__("xgboost").__version__}')
print(f'SHAP: {shap.__version__}')

## 4. Data Loading

### Dataset: Pima Indians Diabetes Database

We use the **Pima Indians Diabetes Database**, originally from the National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK). This dataset was collected to predict whether a patient has diabetes based on diagnostic measurements.

**Dataset characteristics:**
- **768 samples** (female patients, age >= 21)
- **8 features** (clinical measurements)
- **Binary target**: 0 = No Diabetes, 1 = Diabetes
- **Source**: [Kaggle - Pima Indians Diabetes Database](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)

**Features:**
| Feature | Description |
|---------|-------------|
| pregnancies | Number of times pregnant |
| glucose | Plasma glucose concentration (2 hr oral glucose tolerance test) |
| blood_pressure | Diastolic blood pressure (mm Hg) |
| skin_thickness | Triceps skinfold thickness (mm) |
| insulin | 2-Hour serum insulin (mu U/ml) |
| bmi | Body mass index (weight in kg / height in m^2) |
| diabetes_pedigree_function | Diabetes pedigree function (genetic influence) |
| age | Age in years |

In [ ]:
# Load the dataset
DATA_DIR = os.path.join('..', 'Data')
local_path = os.path.join(DATA_DIR, 'diabetes.csv')

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
    print(f'Dataset loaded from: {local_path}')
else:
    print('Downloading Pima Indians Diabetes Dataset...')
    url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
    columns = FEATURE_NAMES + ['outcome']
    df = pd.read_csv(url, header=None, names=columns)
    os.makedirs(DATA_DIR, exist_ok=True)
    df.to_csv(local_path, index=False)
    print(f'Dataset saved to: {local_path}')

# Standardize column names
if 'Outcome' in df.columns:
    df.columns = FEATURE_NAMES + ['outcome']
elif 'outcome' not in df.columns:
    df.columns = FEATURE_NAMES + ['outcome']

print(f'\nDataset shape: {df.shape}')
print(f'Features: {len(FEATURE_NAMES)}')
print(f'Samples: {len(df)}')
df.head(10)

## 5. Exploratory Data Analysis (EDA)

### 5.1 Dataset Overview

In [ ]:
# Basic statistics
print('=== Dataset Info ===')
print(f'Shape: {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print(f'\nData Types:')
print(df.dtypes)
print(f'\nMissing values: {df.isnull().sum().sum()}')
print('\n=== Statistical Summary ===')
df.describe().round(2)

### 5.2 Target Variable Distribution

In [ ]:
# Class distribution
class_counts = df['outcome'].value_counts()
class_pct = df['outcome'].value_counts(normalize=True) * 100

print('Class Distribution:')
print(f'  No Diabetes (0): {class_counts[0]} ({class_pct[0]:.1f}%)')
print(f'  Diabetes    (1): {class_counts[1]} ({class_pct[1]:.1f}%)')
print(f'  Imbalance ratio: {class_counts[0] / class_counts[1]:.2f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['No Diabetes (0)', 'Diabetes (1)'], class_counts.values, color=colors)
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution')
for i, (count, pct) in enumerate(zip(class_counts.values, class_pct.values)):
    axes[0].text(i, count + 10, f'{count}\n({pct:.1f}%)', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=['No Diabetes', 'Diabetes'], colors=colors,
            autopct='%1.1f%%', startangle=90, explode=(0, 0.05))
axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.show()

### 5.3 Feature Distributions

In [ ]:
# Distribution of each feature by class
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, feature in enumerate(FEATURE_NAMES):
    ax = axes[i]
    df[df['outcome'] == 0][feature].hist(ax=ax, bins=25, alpha=0.6, color='#2ecc71', label='No Diabetes')
    df[df['outcome'] == 1][feature].hist(ax=ax, bins=25, alpha=0.6, color='#e74c3c', label='Diabetes')
    ax.set_title(feature.replace('_', ' ').title())
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 5.4 Identifying Missing Values (Zeros)

The Pima dataset uses **0** to represent missing values in certain features where 0 is not physiologically possible (you can't have 0 glucose, 0 blood pressure, 0 BMI, etc.).

In [ ]:
# Check for zero values in features where 0 is not possible
zero_not_possible = ['glucose', 'blood_pressure', 'skin_thickness', 'insulin', 'bmi']

print('Zero values (represent missing data):')
print('-' * 45)
zero_counts = {}
for col in zero_not_possible:
    n_zeros = (df[col] == 0).sum()
    pct = n_zeros / len(df) * 100
    zero_counts[col] = n_zeros
    print(f'  {col:30s}: {n_zeros:4d} zeros ({pct:.1f}%)')

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(list(zero_counts.keys()), list(zero_counts.values()), color='#e67e22')
ax.set_xlabel('Number of Zero Values')
ax.set_title('Missing Values Encoded as Zeros')
for bar, count in zip(bars, zero_counts.values()):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{count} ({count/len(df)*100:.1f}%)', va='center')
plt.tight_layout()
plt.show()

### 5.5 Correlation Analysis

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Correlation with target
print('\nCorrelation with Diabetes Outcome:')
print('-' * 40)
target_corr = df.corr()['outcome'].drop('outcome').sort_values(ascending=False)
for feature, corr_val in target_corr.items():
    print(f'  {feature:35s}: {corr_val:+.3f}')

### 5.6 Box Plots (Outlier Detection)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, feature in enumerate(FEATURE_NAMES):
    ax = axes[i]
    df.boxplot(column=feature, by='outcome', ax=ax)
    ax.set_title(feature.replace('_', ' ').title())
    ax.set_xlabel('Outcome (0=No, 1=Yes)')

plt.suptitle('Feature Distributions by Diabetes Outcome', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Data Preprocessing

### Strategy:
1. Replace zero values (in features where 0 is impossible) with the **median** of non-zero values
2. This is more robust than mean imputation for skewed distributions

In [ ]:
# Preprocessing: replace zeros with median
df_processed = df.copy()

print('PREPROCESSING')
print('-' * 50)

for col in zero_not_possible:
    n_zeros = (df_processed[col] == 0).sum()
    if n_zeros > 0:
        median_val = df_processed[col][df_processed[col] != 0].median()
        df_processed.loc[df_processed[col] == 0, col] = median_val
        print(f'  Replaced {n_zeros} zero values in \'{col}\' with median ({median_val:.1f})')

print(f'\nFinal dataset shape: {df_processed.shape}')
print(f'Class distribution:')
print(f'  No Diabetes (0): {(df_processed["outcome"] == 0).sum()} ({(df_processed["outcome"] == 0).mean()*100:.1f}%)')
print(f'  Diabetes (1):    {(df_processed["outcome"] == 1).sum()} ({(df_processed["outcome"] == 1).mean()*100:.1f}%)')

In [ ]:
# Verify: no more zeros in those columns
print('Zero values after preprocessing:')
for col in zero_not_possible:
    n_zeros = (df_processed[col] == 0).sum()
    print(f'  {col:30s}: {n_zeros} zeros')

print('\nUpdated Statistics:')
df_processed.describe().round(2)

## 7. Train-Test Split & Feature Scaling

In [ ]:
# Separate features and target
X = df_processed[FEATURE_NAMES].values
y = df_processed['outcome'].values

# Stratified split (preserves class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'\nTraining class distribution:')
print(f'  No Diabetes: {(y_train == 0).sum()} ({(y_train == 0).mean()*100:.1f}%)')
print(f'  Diabetes:    {(y_train == 1).sum()} ({(y_train == 1).mean()*100:.1f}%)')

In [ ]:
# Feature Scaling with StandardScaler
# Important: fit ONLY on training data, then transform both train and test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('StandardScaler applied:')
print(f'  Fitted on {X_train.shape[0]} training samples')
print(f'\nScaler parameters (per feature):')
print(f'{"Feature":35s} {"Mean":>10s} {"Std":>10s}')
print('-' * 55)
for name, mean, std in zip(FEATURE_NAMES, scaler.mean_, scaler.scale_):
    print(f'{name:35s} {mean:10.3f} {std:10.3f}')

## 8. Model Training & Comparison

We train and compare three models:

1. **Random Forest** - Ensemble of decision trees with balanced class weights
2. **XGBoost** - Gradient boosting with default hyperparameters
3. **XGBoost (Tuned)** - Gradient boosting with regularization and tuned hyperparameters

Each model is evaluated using:
- **5-Fold Stratified Cross-Validation** on the training set
- **Hold-out test set** evaluation

In [ ]:
# Calculate class weight for XGBoost
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos = n_neg / n_pos
print(f'Class weight ratio (neg/pos): {scale_pos:.2f}')

# Define models
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    'XGBoost (Tuned)': XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=3,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
}

print(f'\nModels to train: {len(models)}')
for name in models:
    print(f'  - {name}')

In [ ]:
# Train and evaluate all models
results = {}
cv = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)

for name, model in models.items():
    print(f'\n{"=" * 50}')
    print(f'Training: {name}')
    print(f'{"=" * 50}')
    
    # Cross-validation on training data
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1')
    print(f'  CV F1 scores: {cv_scores.round(3)}')
    print(f'  CV F1 mean:   {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})')
    
    # Fit on full training set
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Compute metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_val = roc_auc_score(y_test, y_proba)
    
    results[name] = {
        'model': model,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'auc': auc_val,
        'cv_f1_mean': cv_scores.mean(),
        'cv_f1_std': cv_scores.std(),
        'y_pred': y_pred,
        'y_proba': y_proba,
    }
    
    print(f'\n  Test Set Results:')
    print(f'    Accuracy:  {acc:.3f}')
    print(f'    Precision: {prec:.3f}')
    print(f'    Recall:    {rec:.3f}')
    print(f'    F1 Score:  {f1:.3f}')
    print(f'    AUC-ROC:   {auc_val:.3f}')

print(f'\n{"=" * 50}')
print('All models trained successfully!')

## 9. Model Comparison & Selection

In [ ]:
# Results summary table
summary_rows = []
for name, res in results.items():
    summary_rows.append({
        'Model': name,
        'Accuracy': round(res['accuracy'], 4),
        'Precision': round(res['precision'], 4),
        'Recall': round(res['recall'], 4),
        'F1 Score': round(res['f1'], 4),
        'AUC-ROC': round(res['auc'], 4),
        'CV F1 Mean': round(res['cv_f1_mean'], 4),
        'CV F1 Std': round(res['cv_f1_std'], 4),
    })

results_df = pd.DataFrame(summary_rows)
results_df = results_df.set_index('Model')
print('MODEL COMPARISON RESULTS')
print('=' * 60)
display(results_df)

# Select best model by F1
best_name = max(results, key=lambda k: results[k]['f1'])
best_result = results[best_name]
print(f'\nBEST MODEL: {best_name}')
print(f'  F1 Score: {best_result["f1"]:.3f}')
print(f'  AUC-ROC:  {best_result["auc"]:.3f}')

In [ ]:
# Model Comparison Bar Chart
fig, ax = plt.subplots(figsize=(12, 6))
model_names = list(results.keys())
metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
x = np.arange(len(model_names))
width = 0.15
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

for i, metric in enumerate(metrics):
    values = [results[name][metric] for name in model_names]
    ax.bar(x + i * width, values, width, label=metric.upper(), color=colors[i])

ax.set_ylabel('Score')
ax.set_title('Model Comparison - All Metrics')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(model_names)
ax.legend(loc='lower right')
ax.set_ylim(0.5, 1.0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Detailed Evaluation of Best Model

In [ ]:
# Classification Report
print(f'CLASSIFICATION REPORT - {best_name}')
print('=' * 55)
print(classification_report(
    y_test, best_result['y_pred'],
    target_names=['No Diabetes', 'Diabetes'],
    digits=3
))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, best_result['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'],
            annot_kws={'size': 16})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix - {best_name}', fontsize=14)
plt.tight_layout()
plt.show()

print(f'True Negatives:  {cm[0][0]}')
print(f'False Positives: {cm[0][1]}')
print(f'False Negatives: {cm[1][0]}')
print(f'True Positives:  {cm[1][1]}')

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - Model Comparison', fontsize=14)
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall Curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, res in results.items():
    prec_vals, rec_vals, _ = precision_recall_curve(y_test, res['y_proba'])
    ap = average_precision_score(y_test, res['y_proba'])
    ax.plot(rec_vals, prec_vals, label=f'{name} (AP = {ap:.3f})', linewidth=2)

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves - Model Comparison', fontsize=14)
ax.legend(loc='lower left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (from best model)
best_model = best_result['model']

if hasattr(best_model, 'feature_importances_'):
    fig, ax = plt.subplots(figsize=(10, 6))
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    sorted_features = [FEATURE_NAMES[i] for i in indices]
    sorted_importances = importances[indices]
    
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(sorted_features)))
    ax.barh(range(len(sorted_features)), sorted_importances[::-1], color=colors)
    ax.set_yticks(range(len(sorted_features)))
    ax.set_yticklabels(sorted_features[::-1])
    ax.set_xlabel('Feature Importance', fontsize=12)
    ax.set_title(f'Feature Importance - {best_name}', fontsize=14)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print('Feature Importance Ranking:')
    for i, (feat, imp) in enumerate(zip(sorted_features, sorted_importances)):
        print(f'  {i+1}. {feat:35s}: {imp:.4f}')

## 11. Explainability with SHAP

**SHAP (SHapley Additive exPlanations)** provides a unified approach to explain the output of any machine learning model. It uses game theory (Shapley values) to assign each feature an importance value for a particular prediction.

### Why SHAP?
- **Global explanations**: Which features matter most overall
- **Local explanations**: Why a specific prediction was made
- **Consistent & theoretically grounded**: Based on Shapley values from cooperative game theory

In [ ]:
# SHAP Analysis
print('Computing SHAP values...')
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_scaled)

# For binary classifiers, shap_values may be [class0, class1]
if isinstance(shap_values, list):
    shap_vals = shap_values[1]  # positive class (diabetes)
else:
    shap_vals = shap_values

print(f'SHAP values shape: {shap_vals.shape}')
print('Done!')

In [ ]:
# SHAP Summary Plot (Beeswarm)
# Shows each feature's impact on the model output
# Red = high feature value, Blue = low feature value
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_test_scaled, feature_names=FEATURE_NAMES, show=False)
plt.title('SHAP Summary Plot - Feature Impact on Diabetes Prediction')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Feature Importance (Bar Plot)
# Mean absolute SHAP values per feature
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_test_scaled, feature_names=FEATURE_NAMES, plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Mean |SHAP|)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Waterfall Plot for a single prediction
# Shows how each feature contributed to pushing the prediction
# from the base value (average prediction) to the final output

# Pick a high-risk sample
high_risk_idx = np.argmax(best_result['y_proba'])
print(f'Explaining prediction for test sample #{high_risk_idx}')
print(f'  Predicted probability: {best_result["y_proba"][high_risk_idx]:.3f}')
print(f'  Actual label: {"Diabetes" if y_test[high_risk_idx] == 1 else "No Diabetes"}')
print(f'\n  Feature values:')
for name, val in zip(FEATURE_NAMES, X_test[high_risk_idx]):
    print(f'    {name:35s}: {val:.1f}')

# Waterfall
expected = explainer.expected_value
if isinstance(expected, list):
    expected = expected[1]

explanation = shap.Explanation(
    values=shap_vals[high_risk_idx],
    base_values=expected,
    data=X_test_scaled[high_risk_idx],
    feature_names=FEATURE_NAMES
)
plt.figure(figsize=(10, 6))
shap.waterfall_plot(explanation, show=False)
plt.title(f'SHAP Waterfall - High Risk Sample')
plt.tight_layout()
plt.show()

## 12. Save Model Bundle

The model is saved as a **bundle dictionary** containing:
- `model`: The trained sklearn/xgboost model
- `scaler`: The fitted StandardScaler (for preprocessing new data)
- `feature_names`: List of feature names (to ensure correct input ordering)

This bundle is loaded by the FastAPI backend at runtime.

In [ ]:
# Save model bundle
bundle = {
    'model': best_model,
    'scaler': scaler,
    'feature_names': FEATURE_NAMES,
}

# Save to backend data directory
model_output = os.path.join('..', 'app', 'backend', 'data', 'best_model.pkl')
os.makedirs(os.path.dirname(model_output), exist_ok=True)
with open(model_output, 'wb') as f:
    pickle.dump(bundle, f)
print(f'Model bundle saved to: {model_output}')

# Save copy to Models directory
model_copy = os.path.join('..', 'Models', 'best_model.pkl')
os.makedirs(os.path.dirname(model_copy), exist_ok=True)
with open(model_copy, 'wb') as f:
    pickle.dump(bundle, f)
print(f'Model bundle copy saved to: {model_copy}')

print(f'\nBundle contents:')
print(f'  Model:    {type(best_model).__name__}')
print(f'  Scaler:   StandardScaler')
print(f'  Features: {len(FEATURE_NAMES)} ({FEATURE_NAMES})')

In [ ]:
# Verify the saved bundle loads correctly
with open(model_output, 'rb') as f:
    loaded_bundle = pickle.load(f)

loaded_model = loaded_bundle['model']
loaded_scaler = loaded_bundle['scaler']
loaded_features = loaded_bundle['feature_names']

# Test prediction with loaded model
test_sample = X_test[:1]  # take first test sample
test_scaled = loaded_scaler.transform(test_sample)
test_pred = loaded_model.predict_proba(test_scaled)[0]

print('Bundle verification:')
print(f'  Model type: {type(loaded_model).__name__}')
print(f'  Features:   {loaded_features}')
print(f'  Test prediction: No Diabetes={test_pred[0]:.3f}, Diabetes={test_pred[1]:.3f}')
print('\n  Bundle loads and predicts correctly!')

## 13. Conclusion

### Summary

In this notebook, we built a complete machine learning pipeline for **Type 2 Diabetes early detection**:

1. **Dataset**: Pima Indians Diabetes Database (768 samples, 8 features)
2. **Preprocessing**: Median imputation for missing values (encoded as zeros), StandardScaler normalization
3. **Models trained**: Random Forest, XGBoost, XGBoost (Tuned)
4. **Best model selected by F1 Score** to balance precision and recall
5. **Explainability**: SHAP analysis provides transparent, interpretable predictions
6. **Deployment**: Model saved as a bundle for the FastAPI backend

### Key Findings

- **Glucose** is the most important feature for diabetes prediction
- **BMI** and **Age** are strong secondary predictors
- The model achieves reasonable discrimination (AUC > 0.80) on this dataset
- Feature scaling with StandardScaler ensures consistent input processing

### Integration with Insulyn AI

The saved model bundle is loaded by the FastAPI backend (`app/backend/app/main.py`) which:
- Unpacks the model, scaler, and feature names
- Applies the scaler to incoming predictions
- Returns risk classification, probability, feature importances, and SHAP explanations
- Provides a REST API consumed by the React frontend